# Mid-Trigger-Ratio Sweep on GSM8K

扫描 `mid_trigger_ratio` 参数（0.1 ~ 0.9，步长 0.05）对 GSM8K 评测结果的影响。

**核心设计：任务池 + GPU 池**
- 将所有 `mid_trigger_ratio` 配置排入任务队列
- 可用 GPU 组成 GPU 池
- 每个 GPU 从任务池中取一个任务执行，完成后自动取下一个
- 适用于 GPU 数量少于任务数量的场景（17 个任务，N 个 GPU）

**核心指标：**
- GSM8K 准确率 vs mid_trigger_ratio
- Total NFE vs mid_trigger_ratio
- 每步解码 token 数量分布

## 1. 环境设置

In [ ]:
import os
import torch
import gc

os.environ['CUDA_VISIBLE_DEVICES'] = '0,1,2,3,4,5,6,7'

os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['HF_ALLOW_CODE_EVAL'] = '1'
os.environ['HF_DATASETS_TRUST_REMOTE_CODE'] = 'true'

os.chdir('llada')

os.makedirs('nlogs', exist_ok=True)

torch.cuda.empty_cache()
gc.collect()

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} "
          f"({torch.cuda.get_device_properties(i).total_mem / 1024**3:.1f} GB)")

## 2. 扫参配置 & 任务池 / GPU 池

- `SWEEP_VALUES`: 0.1 ~ 0.9，步长 0.05（共 17 个值）
- `GPU_POOL`: 可用的 GPU 编号列表
- 调度逻辑：每个 GPU 从任务队列取任务，跑完一个再取下一个，直到队列为空

In [ ]:
import subprocess
import datetime
import threading
import queue
import numpy as np

# ========== 扫参配置 ==========
SWEEP_VALUES = list(np.round(np.arange(0.1, 0.95, 0.05), 2))

# ========== GPU 池 ==========
GPU_POOL = [0, 1, 2, 3, 4, 5, 6, 7]  # 根据实际可用 GPU 修改

# ========== 评测参数（与 expand_rewarm_fb_on 配置完全一致） ==========
task = "gsm8k"
fewshot = 5
limit = None      # 全量 GSM8K（1319 题）
seed = 42
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

gen_length = 256
steps = 256
block_length = 32
threshold = 0.9

# 固定配置（除 mid_trigger_ratio 外，与 expand_rewarm_fb_on 一致）
FIXED_DUAL_CACHE = True
FIXED_MID_BLOCK_EXPAND = True
FIXED_REWARM = True               # rewarm_on_expand=True
FIXED_FALLBACK = True              # front_block_fallback_only=True

print(f"Sweep mid_trigger_ratio: {SWEEP_VALUES}")
print(f"Total tasks: {len(SWEEP_VALUES)}")
print(f"GPU pool: {GPU_POOL} ({len(GPU_POOL)} GPUs)")
print(f"limit: {limit} ({'全量' if limit is None else f'{limit} 题'})")
print(f"Timestamp: {timestamp}")

## 3. 任务池调度：启动所有实验

In [ ]:
task_queue = queue.Queue()
for ratio in SWEEP_VALUES:
    task_queue.put(ratio)

results_lock = threading.Lock()
all_results = []  # (ratio, name, log_file, output_dir, return_code)


def gpu_worker(gpu_id):
    """GPU worker: 不断从任务队列取任务，直到队列为空"""
    while True:
        try:
            ratio = task_queue.get_nowait()
        except queue.Empty:
            return

        name = f"ratio_{ratio:.2f}"
        log_file = f"nlogs/sweep_{task}_{name}_{timestamp}.log"
        output_dir = f"evals_results/sweep_ratio/{task}-{name}-{timestamp}"
        records_dir = f"{output_dir}/step_records"

        extra_args = [
            f"dual_cache={FIXED_DUAL_CACHE}",
            f"mid_block_expand={FIXED_MID_BLOCK_EXPAND}",
            f"mid_trigger_ratio={ratio}",
            f"rewarm_on_expand={FIXED_REWARM}",
            f"front_block_fallback_only={FIXED_FALLBACK}",
        ]

        base_args = [
            f"model_path='GSAI-ML/LLaDA-8B-Instruct'",
            f"gen_length={gen_length}",
            f"steps={steps}",
            f"block_length={block_length}",
            f"threshold={threshold}",
            "use_cache=True",
            "show_speed=True",
            f"step_records_dir='{records_dir}'",
            f"seed={seed}",
        ]
        all_args = base_args + extra_args
        model_args_str = ",".join(all_args)

        limit_str = f" --limit {limit}" if limit is not None else ""
        cmd = (
            f"CUDA_VISIBLE_DEVICES={gpu_id} accelerate launch eval_llada.py "
            f"--tasks {task} --num_fewshot {fewshot}{limit_str} "
            f"--confirm_run_unsafe_code --model llada_dist "
            f"--model_args {model_args_str} "
            f"--output_path {output_dir} --log_samples"
        )

        print(f"[GPU {gpu_id}] START  mid_trigger_ratio={ratio:.2f}")

        p = subprocess.Popen(
            cmd, shell=True,
            stdout=open(log_file, "w"),
            stderr=subprocess.STDOUT,
        )
        rc = p.wait()

        status = "OK" if rc == 0 else f"FAILED(exit={rc})"
        print(f"[GPU {gpu_id}] DONE   mid_trigger_ratio={ratio:.2f}  {status}")

        with results_lock:
            all_results.append((ratio, name, log_file, output_dir, rc))

        task_queue.task_done()


# 启动 GPU worker 线程
threads = []
for gpu_id in GPU_POOL:
    t = threading.Thread(target=gpu_worker, args=(gpu_id,), daemon=True)
    t.start()
    threads.append(t)

print(f"\nLaunched {len(threads)} GPU workers for {len(SWEEP_VALUES)} tasks.")
print("Waiting for all tasks to complete...")

In [ ]:
# 等待所有线程完成
for t in threads:
    t.join()

print(f"\nAll {len(all_results)} / {len(SWEEP_VALUES)} tasks finished.")

# 按 ratio 排序
all_results.sort(key=lambda x: x[0])

for ratio, name, log_file, output_dir, rc in all_results:
    status = "OK" if rc == 0 else f"FAILED(exit={rc})"
    print(f"  ratio={ratio:.2f}  {status}  log={log_file}")

## 4. 解析评测结果

In [ ]:
import glob
import re
import json
import pandas as pd

log_files = sorted(glob.glob(f"nlogs/sweep_{task}_ratio_*_{timestamp}.log"))

print(f"Mid-Trigger-Ratio Sweep GSM8K Results (timestamp={timestamp})")
print("=" * 90)

parsed_results = []
for log_file in log_files:
    fname = os.path.basename(log_file)
    name_part = fname.replace(f"sweep_{task}_", "").replace(f"_{timestamp}.log", "")
    ratio_val = float(name_part.replace("ratio_", ""))

    with open(log_file, 'r') as f:
        content = f.read()

    acc_match = re.search(r'exact_match.*?[\|,]\s*[\|]?\s*([\d.]+)', content)
    acc = float(acc_match.group(1)) if acc_match else None

    speed_match = re.search(r'Tokens per second:\s*([\d.]+)', content)
    speed = float(speed_match.group(1)) if speed_match else None

    nfe_match = re.search(r'Total NFE is (\d+)', content)
    nfe = int(nfe_match.group(1)) if nfe_match else None

    time_match = re.search(r'Total time taken:\s*([\d.]+)', content)
    time_sec = float(time_match.group(1)) if time_match else None

    parsed_results.append({
        'ratio': ratio_val,
        'config': name_part,
        'accuracy': acc,
        'tokens_per_sec': speed,
        'total_nfe': nfe,
        'time_sec': time_sec,
        'log_file': log_file,
    })

parsed_results.sort(key=lambda x: x['ratio'])

print(f"{'Ratio':<10} {'Acc':<10} {'Tok/s':<12} {'Total NFE':<12} {'Time(s)':<10}")
print("-" * 55)
for r in parsed_results:
    acc_str = f"{r['accuracy']:.4f}" if r['accuracy'] is not None else "N/A"
    speed_str = f"{r['tokens_per_sec']:.1f}" if r['tokens_per_sec'] is not None else "N/A"
    nfe_str = str(r['total_nfe']) if r['total_nfe'] is not None else "N/A"
    time_str = f"{r['time_sec']:.1f}" if r['time_sec'] is not None else "N/A"
    print(f"{r['ratio']:<10.2f} {acc_str:<10} {speed_str:<12} {nfe_str:<12} {time_str:<10}")

df_summary = pd.DataFrame(parsed_results)
display(df_summary)

## 5. 加载 Step Records

In [ ]:
step_data = {}

for r in parsed_results:
    name = r['config']
    out_dir = f"evals_results/sweep_ratio/{task}-{name}-{timestamp}"
    records_path = os.path.join(out_dir, 'step_records', 'step_records.json')
    if os.path.exists(records_path):
        with open(records_path, 'r') as f:
            step_data[name] = json.load(f)
        print(f"Loaded {name}: {len(step_data[name])} samples, "
              f"total steps = {sum(len(s) for s in step_data[name])}")
    else:
        print(f"WARNING: {records_path} not found")

print(f"\nLoaded step records for {len(step_data)} / {len(parsed_results)} configs.")

## 6. Accuracy & NFE vs mid_trigger_ratio（核心图表）

In [ ]:
import matplotlib.pyplot as plt

ratios = [r['ratio'] for r in parsed_results if r['accuracy'] is not None]
accs   = [r['accuracy'] for r in parsed_results if r['accuracy'] is not None]
nfes   = [r['total_nfe'] for r in parsed_results if r['total_nfe'] is not None]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# --- Accuracy vs mid_trigger_ratio ---
ax1.plot(ratios, accs, 'o-', color='#2196F3', linewidth=2, markersize=6)
ax1.set_xlabel('mid_trigger_ratio', fontsize=12)
ax1.set_ylabel('GSM8K Accuracy', fontsize=12)
ax1.set_title('Accuracy vs mid_trigger_ratio', fontsize=14, fontweight='bold')
ax1.set_xticks(ratios)
ax1.set_xticklabels([f'{r:.2f}' for r in ratios], rotation=45, ha='right', fontsize=9)
ax1.grid(True, alpha=0.3)
for x, y in zip(ratios, accs):
    ax1.annotate(f'{y:.3f}', (x, y), textcoords='offset points',
                 xytext=(0, 8), ha='center', fontsize=7)

# --- NFE vs mid_trigger_ratio ---
nfe_ratios = [r['ratio'] for r in parsed_results if r['total_nfe'] is not None]
ax2.plot(nfe_ratios, nfes, 's-', color='#FF5722', linewidth=2, markersize=6)
ax2.set_xlabel('mid_trigger_ratio', fontsize=12)
ax2.set_ylabel('Total NFE', fontsize=12)
ax2.set_title('NFE vs mid_trigger_ratio', fontsize=14, fontweight='bold')
ax2.set_xticks(nfe_ratios)
ax2.set_xticklabels([f'{r:.2f}' for r in nfe_ratios], rotation=45, ha='right', fontsize=9)
ax2.grid(True, alpha=0.3)
for x, y in zip(nfe_ratios, nfes):
    ax2.annotate(str(y), (x, y), textcoords='offset points',
                 xytext=(0, 8), ha='center', fontsize=7)

fig.suptitle('Mid-Trigger-Ratio Sweep on GSM8K', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
os.makedirs('../eval_results', exist_ok=True)
plt.savefig('../eval_results/sweep_ratio_acc_nfe.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: eval_results/sweep_ratio_acc_nfe.png")

## 7. Accuracy-NFE Trade-off 散点图

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

valid = [r for r in parsed_results if r['accuracy'] is not None and r['total_nfe'] is not None]
xs = [r['total_nfe'] for r in valid]
ys = [r['accuracy'] for r in valid]
labels = [f"{r['ratio']:.2f}" for r in valid]

sc = ax.scatter(xs, ys, c=[r['ratio'] for r in valid], cmap='viridis',
                s=80, edgecolors='gray', zorder=5)
plt.colorbar(sc, ax=ax, label='mid_trigger_ratio')

for x, y, lbl in zip(xs, ys, labels):
    ax.annotate(lbl, (x, y), textcoords='offset points',
                xytext=(6, 6), fontsize=8, alpha=0.8)

ax.set_xlabel('Total NFE', fontsize=12)
ax.set_ylabel('GSM8K Accuracy', fontsize=12)
ax.set_title('Accuracy vs NFE Trade-off (colored by mid_trigger_ratio)',
             fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../eval_results/sweep_ratio_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: eval_results/sweep_ratio_tradeoff.png")

## 8. 每步解码 Token 分布（选取部分 ratio 可视化）

In [ ]:
from collections import defaultdict

# 选几个有代表性的 ratio 画图
sample_ratios = [0.1, 0.3, 0.5, 0.7, 0.9]
sample_keys = [f"ratio_{r:.2f}" for r in sample_ratios if f"ratio_{r:.2f}" in step_data]

if sample_keys:
    n_plots = len(sample_keys)
    fig, axes = plt.subplots(1, n_plots, figsize=(5 * n_plots, 5), sharey=True)
    if n_plots == 1:
        axes = [axes]

    cmap = plt.cm.viridis
    for ax_idx, cname in enumerate(sample_keys):
        ax = axes[ax_idx]
        samples = step_data[cname]

        step_transferred = defaultdict(list)
        for sample_records in samples:
            for rec in sample_records:
                step_transferred[rec['global_step']].append(rec['transferred'])

        max_step = max(step_transferred.keys()) if step_transferred else 0
        steps_range = list(range(max_step + 1))
        means = [np.mean(step_transferred[s]) if s in step_transferred else 0 for s in steps_range]

        ratio_val = float(cname.replace('ratio_', ''))
        color = cmap(ratio_val)
        ax.bar(steps_range, means, color=color, alpha=0.7)
        ax.set_xlabel('Global Step')

        res = next((r for r in parsed_results if r['config'] == cname), None)
        acc_str = f"Acc={res['accuracy']:.4f}" if res and res['accuracy'] else ""
        nfe_str = f"NFE={res['total_nfe']}" if res and res['total_nfe'] else ""
        ax.set_title(f"ratio={ratio_val:.2f}\n{acc_str}  {nfe_str}", fontsize=10)
        ax.set_xlim(-0.5, max_step + 0.5)

    axes[0].set_ylabel('Tokens Transferred')
    fig.suptitle('Per-Step Decoded Tokens (selected ratios)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../eval_results/sweep_ratio_per_step.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: eval_results/sweep_ratio_per_step.png")
else:
    print("No step_data available for visualization.")

## 8.5 汇总表 & 三指标折线图（ACC / NFE / Time vs ratio）

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 从 parsed_results 直接构建 DataFrame
df = pd.DataFrame(parsed_results)
df = df.dropna(subset=['accuracy', 'total_nfe', 'time_sec'])
df = df.sort_values('ratio').reset_index(drop=True)

# Tab 分隔汇总表（可直接粘贴到 Excel / Google Sheets）
print("ratio\tACC\tNFE\tTime(s)\tTok/s")
for _, row in df.iterrows():
    tok_s = f"{row['tokens_per_sec']:.1f}" if pd.notna(row['tokens_per_sec']) else ""
    print(f"{row['ratio']:.2f}\t{row['accuracy']:.4f}\t"
          f"{int(row['total_nfe'])}\t{row['time_sec']:.1f}\t{tok_s}")

# 三合一折线图
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ACC vs ratio
ax = axes[0]
ax.plot(df['ratio'], df['accuracy'], 'o-', color='#2196F3', lw=2, ms=6)
ax.set_xlabel('mid_trigger_ratio')
ax.set_ylabel('Accuracy')
ax.set_title('ACC vs mid_trigger_ratio', fontweight='bold')
ax.grid(True, alpha=0.3)
for _, row in df.iterrows():
    ax.annotate(f"{row['accuracy']:.4f}", (row['ratio'], row['accuracy']),
                textcoords='offset points', xytext=(0, 8), ha='center', fontsize=7)

# NFE vs ratio
ax = axes[1]
ax.plot(df['ratio'], df['total_nfe'], 's-', color='#FF5722', lw=2, ms=6)
ax.set_xlabel('mid_trigger_ratio')
ax.set_ylabel('Total NFE')
ax.set_title('NFE vs mid_trigger_ratio', fontweight='bold')
ax.grid(True, alpha=0.3)
for _, row in df.iterrows():
    ax.annotate(f"{int(row['total_nfe'])}", (row['ratio'], row['total_nfe']),
                textcoords='offset points', xytext=(0, 8), ha='center', fontsize=7)

# Time vs ratio
ax = axes[2]
ax.plot(df['ratio'], df['time_sec'], 'D-', color='#4CAF50', lw=2, ms=6)
ax.set_xlabel('mid_trigger_ratio')
ax.set_ylabel('Time (s)')
ax.set_title('Time vs mid_trigger_ratio', fontweight='bold')
ax.grid(True, alpha=0.3)
for _, row in df.iterrows():
    ax.annotate(f"{row['time_sec']:.0f}", (row['ratio'], row['time_sec']),
                textcoords='offset points', xytext=(0, 8), ha='center', fontsize=7)

fig.suptitle('Mid-Trigger-Ratio Sweep Summary (GSM8K, full 1319 samples)',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
os.makedirs('../eval_results', exist_ok=True)
plt.savefig('../eval_results/sweep_ratio_summary_3in1.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: eval_results/sweep_ratio_summary_3in1.png")

display(df[['ratio', 'accuracy', 'total_nfe', 'time_sec', 'tokens_per_sec']])

## 9. 从已有结果重新加载（跨机器 / 补跑用）

将 `evals_results/sweep_ratio/` 和 `nlogs/` 目录拷贝过来后，修改下方 `timestamp` 或让其自动检测。

In [ ]:
import glob, re, json, os
import numpy as np
import pandas as pd

task = "gsm8k"
gen_length = 256

SWEEP_VALUES = list(np.round(np.arange(0.1, 0.95, 0.05), 2))
config_names = [f"ratio_{r:.2f}" for r in SWEEP_VALUES]

# 自动检测最新 timestamp
latest_logs = sorted(
    glob.glob(f"nlogs/sweep_{task}_{config_names[0]}_*.log"),
    key=os.path.getmtime, reverse=True,
)
if latest_logs:
    fname = os.path.basename(latest_logs[0])
    timestamp = fname.replace(f"sweep_{task}_{config_names[0]}_", "").replace(".log", "")
    print(f"Auto-detected latest timestamp: {timestamp}")
else:
    timestamp = "NOTFOUND"
    print("WARNING: No log found!")

# 加载 step_data
step_data = {}
missing_step_data = []
for name in config_names:
    rpath = f"evals_results/sweep_ratio/{task}-{name}-{timestamp}/step_records/step_records.json"
    if os.path.exists(rpath):
        with open(rpath, 'r') as f:
            step_data[name] = json.load(f)
        print(f"  [OK]  {name}: {len(step_data[name])} samples, "
              f"{sum(len(s) for s in step_data[name])} total steps")
    else:
        missing_step_data.append(name)
        print(f"  [MISS] {name}: step_records.json NOT FOUND")

# 加载 parsed_results（从 log 文件提取 accuracy / NFE）
parsed_results = []
missing_logs = []
for name in config_names:
    log_file = f"nlogs/sweep_{task}_{name}_{timestamp}.log"
    if not os.path.exists(log_file):
        missing_logs.append(name)
        continue
    with open(log_file, 'r') as f:
        content = f.read()

    ratio_val = float(name.replace("ratio_", ""))
    acc_match = re.search(r'exact_match.*?[\|,]\s*[\|]?\s*([\d.]+)', content)
    nfe_match = re.search(r'Total NFE is (\d+)', content)
    time_match = re.search(r'Total time taken:\s*([\d.]+)', content)
    speed_match = re.search(r'Tokens per second:\s*([\d.]+)', content)

    parsed_results.append({
        'ratio': ratio_val,
        'config': name,
        'accuracy': float(acc_match.group(1)) if acc_match else None,
        'total_nfe': int(nfe_match.group(1)) if nfe_match else None,
        'time_sec': float(time_match.group(1)) if time_match else None,
        'tokens_per_sec': float(speed_match.group(1)) if speed_match else None,
    })

parsed_results.sort(key=lambda x: x['ratio'])

# ===== 汇总报告 =====
print(f"\n{'='*70}")
print(f"Loaded {len(step_data)} step_data, {len(parsed_results)} parsed_results")
if missing_logs:
    print(f"  Missing logs: {missing_logs}")
if missing_step_data:
    print(f"  Missing step_records: {missing_step_data}")

# 打印 parsed_results 表格（所有 ratio 的 accuracy/NFE）
print(f"\n{'Ratio':<10} {'Acc':<10} {'NFE':<12} {'Tok/s':<12} {'Time(s)':<10} {'StepRec':<8}")
print("-" * 62)
for r in parsed_results:
    acc_str = f"{r['accuracy']:.4f}" if r['accuracy'] is not None else "N/A"
    nfe_str = str(r['total_nfe']) if r['total_nfe'] is not None else "N/A"
    speed_str = f"{r['tokens_per_sec']:.1f}" if r['tokens_per_sec'] is not None else "N/A"
    time_str = f"{r['time_sec']:.1f}" if r['time_sec'] is not None else "N/A"
    has_step = "YES" if r['config'] in step_data else "NO"
    print(f"{r['ratio']:<10.2f} {acc_str:<10} {nfe_str:<12} {speed_str:<12} {time_str:<10} {has_step:<8}")

print(f"\nNow re-run Section 6~8 cells to regenerate all plots.")